# FEMA OpenFEMA API — Exploration Notebook

This notebook walks through the FEMA OpenFEMA API, focusing on disaster declarations.

**API base URL:** `https://www.fema.gov/api/open/v2/disasterDeclarationsSummaries`  
**Docs:** https://www.fema.gov/about/openfema/data-sets  
**API reference:** https://www.fema.gov/api/open/v2/disasterDeclarationsSummaries

In [5]:
import requests
import pandas as pd
import json

BASE_URL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"

## 1. Inspect the API metadata

OpenFEMA uses OData-style query parameters: `$filter`, `$top`, `$skip`, `$orderby`, `$select`.

In [8]:
# Fetch just 1 record to see the full schema
params = {
    "$format": "json",
    "$top": 1,
    "$inlinecount": "allpages",
}

r = requests.get(BASE_URL, params=params)
r.raise_for_status()
data = r.json()

print("Total records in dataset:", data["metadata"]["count"])   
print("All available fields:")
print(list(data["DisasterDeclarationsSummaries"][0].keys()))

Total records in dataset: 69769
All available fields:
['femaDeclarationString', 'disasterNumber', 'state', 'declarationType', 'declarationDate', 'fyDeclared', 'incidentType', 'declarationTitle', 'ihProgramDeclared', 'iaProgramDeclared', 'paProgramDeclared', 'hmProgramDeclared', 'incidentBeginDate', 'incidentEndDate', 'disasterCloseoutDate', 'tribalRequest', 'fipsStateCode', 'fipsCountyCode', 'placeCode', 'designatedArea', 'declarationRequestNumber', 'lastIAFilingDate', 'incidentId', 'region', 'designatedIncidentTypes', 'lastRefresh', 'hash', 'id']


In [ ]:
print(json.dumps(data["DisasterDeclarationsSummaries"][0], indent=2))

{
  "femaDeclarationString": "FM-5529-OR",
  "disasterNumber": 5529,
  "state": "OR",
  "declarationType": "FM",
  "declarationDate": "2024-08-09T00:00:00.000Z",
  "fyDeclared": 2024,
  "incidentType": "Fire",
  "declarationTitle": "LEE FALLS FIRE",
  "ihProgramDeclared": false,
  "iaProgramDeclared": false,
  "paProgramDeclared": true,
  "hmProgramDeclared": true,
  "incidentBeginDate": "2024-08-08T00:00:00.000Z",
  "incidentEndDate": null,
  "disasterCloseoutDate": null,
  "tribalRequest": false,
  "fipsStateCode": "41",
  "fipsCountyCode": "067",
  "placeCode": "99067",
  "designatedArea": "Washington (County)",
  "declarationRequestNumber": "24122",
  "lastIAFilingDate": null,
  "incidentId": "2024081001",
  "region": 10,
  "designatedIncidentTypes": "R",
  "lastRefresh": "2024-08-27T18:22:14.800Z",
  "hash": "ae87cf3c6ed795015b714af7166c7c295b2b67c7",
  "id": "09e3f81a-5e16-4b72-b317-1c64e0cfa59c"
}


## 2. What incident types exist?

In [9]:
# Fetch a larger sample to see all incident types
params = {
    "$format": "json",
    "$top": 1000,
    "$select": "incidentType",
}

r = requests.get(BASE_URL, params=params)
r.raise_for_status()
records = r.json()["DisasterDeclarationsSummaries"]
types = pd.DataFrame(records)["incidentType"].value_counts()
print(types.to_string())

incidentType
Fire                   576
Severe Storm           166
Flood                  127
Hurricane               67
Winter Storm            32
Tornado                 26
Biological               4
Straight-Line Winds      2


## 3. Fetch all earthquake declarations

In [ ]:
def fetch_all_pages(base_url: str, extra_params: dict) -> pd.DataFrame:
    """Paginate through all OpenFEMA results and return a DataFrame."""
    frames = []
    skip = 0
    page_size = 1000
    total = None

    while True:
        params = {
            "$format": "json",
            "$top": page_size,
            "$skip": skip,
            "$inlinecount": "allpages",
            **extra_params,
        }
        r = requests.get(base_url, params=params)
        r.raise_for_status()
        data = r.json()

        key = [k for k in data.keys() if k != "metadata"][0]
        records = data[key]
        if not records:
            break

        frames.append(pd.DataFrame(records))

        if total is None:
            total = int(data["metadata"]["count"])
            print(f"Total matching records: {total}")

        skip += page_size
        if skip >= total:
            break

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


df_eq = fetch_all_pages(BASE_URL, {
    "$filter": "incidentType eq 'Earthquake'"
})

print(f"\nFetched {len(df_eq)} earthquake declaration records")
df_eq.head()

In [ ]:
df_eq.dtypes

In [ ]:
df_eq.isna().mean().sort_values(ascending=False).to_frame("pct_missing")

## 4. Explore the earthquake declarations

In [ ]:
# Parse dates
date_cols = ["declarationDate", "incidentBeginDate", "incidentEndDate"]
for col in date_cols:
    df_eq[col] = pd.to_datetime(df_eq[col], utc=True, errors="coerce")

df_eq["declaration_year"] = df_eq["declarationDate"].dt.year
df_eq[["declarationDate", "incidentBeginDate", "incidentEndDate", "state"]].head()

In [ ]:
# Which states have the most earthquake declarations?
df_eq["state"].value_counts().head(15)

In [ ]:
# Declarations per year
df_eq["declaration_year"].value_counts().sort_index()

In [ ]:
# Which federal programs were activated?
program_cols = ["ihProgramDeclared", "iaProgramDeclared", "paProgramDeclared", "hmProgramDeclared"]
for col in program_cols:
    df_eq[col] = pd.to_numeric(df_eq[col], errors="coerce").fillna(0).astype(int)

df_eq[program_cols].sum().to_frame("times_activated")

In [ ]:
# How long do earthquake disasters typically last?
df_eq["duration_days"] = (df_eq["incidentEndDate"] - df_eq["incidentBeginDate"]).dt.days
df_eq["duration_days"].describe()

In [ ]:
# Declaration lag: how many days after the earthquake was the declaration made?
df_eq["declaration_lag_days"] = (df_eq["declarationDate"] - df_eq["incidentBeginDate"]).dt.days
df_eq["declaration_lag_days"].describe()

## 5. Filter by date range (matching USGS query window)

In [ ]:
df_recent = fetch_all_pages(BASE_URL, {
    "$filter": (
        "incidentType eq 'Earthquake' and "
        "declarationDate ge '2015-01-01T00:00:00.000z' and "
        "declarationDate le '2024-12-31T23:59:59.000z'"
    )
})

print(f"Records 2015–2024: {len(df_recent)}")
df_recent[["declarationDate", "state", "declarationTitle", "designatedArea"]].head(10)

## 6. Explore the Public Assistance dataset

This gives dollar amounts per disaster — useful as a direct measure of economic impact.

In [ ]:
PA_URL = "https://www.fema.gov/api/open/v1/PublicAssistanceFundedProjectsDetails"

# Peek at schema with 1 record
r = requests.get(PA_URL, params={"$format": "json", "$top": 1})
r.raise_for_status()
pa_data = r.json()

key = [k for k in pa_data.keys() if k != "metadata"][0]
print("PA fields:", list(pa_data[key][0].keys()))

In [ ]:
# Get PA records for earthquake disasters in CA
df_pa = fetch_all_pages(PA_URL, {
    "$filter": "state eq 'CA'",
    "$select": "disasterNumber,state,county,projectAmount,federalShareObligated,incidentType",
})

print(f"CA PA records: {len(df_pa)}")
df_pa.head()

## 7. Notes for the project

**Key fields for merging with USGS:**
- `state` — two-letter abbreviation, matches what we'll extract from USGS `place`
- `incidentBeginDate` — closest proxy to the earthquake date (for time-window matching)
- `declarationDate` — when the federal response was triggered

**Key fields for impact analysis:**
- `paProgramDeclared` — Public Assistance activated (infrastructure damage)
- `iaProgramDeclared` — Individual Assistance activated (personal property damage)
- `declaration_lag_days` — how fast the federal government responded

**Important caveats:**
- There are only ~200 total earthquake declarations in the full FEMA dataset — they are rare
- Most earthquakes, even large ones, never trigger a federal declaration
- A low match rate (~3%) with USGS is expected and scientifically meaningful
- The 90-day merge window is a key modeling decision we should justify in the report